# Experiment 8: Extended Training --- 40 epochs + Cosine LR Decay

## Rationale

All 7 prior experiments trained for 15 epochs. E1's training loss was still decreasing at epoch 15. Extended training with cosine LR decay tests whether additional optimization can push past the ~92.5% ceiling.

**Single variable changed**: training schedule --- 40 epochs with `CosineAnnealingLR(T_max=40)`
**Held constant**: architecture (DiagnosticCNN), loss (CrossEntropy), optimizer (Adam lr=0.001), data (no augmentation)

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Import Libraries | Load PyTorch, src modules, detect device | --- |
| 2 | Load Dataset | FashionMNIST with train/test split | `src/data_utils.py` |
| 3 | Define DiagnosticCNN | Identical architecture to E1 baseline | --- |
| 4 | Train Extended Schedule | Train 40 epochs with CosineAnnealingLR | `src/train_utils.py` |
| 5 | Evaluate Model | Per-class TPR, Precision, confusion matrix | `src/eval_utils.py` |
| 6 | ROC & PR Curves | ROC-AUC and PR-AUC scores | `src/eval_utils.py`, `src/vis_utils.py` |
| 7 | Compare with E1 | Side-by-side metrics vs 15-epoch baseline | `src/eval_utils.py` |
| 8 | Save Outputs | Save metrics to outputs/error_analysis/extended_training/ | --- |

---


In [ ]:
import os, sys
# Detect project root: look for src/ directory in CWD or parents
def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from src.data_utils import load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores)

OUT_DIR = '../outputs/error_analysis/extended_training'
os.makedirs(OUT_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'
print(f"Device: {device}")

PyTorch: 2.13.0+cu130
Device: cuda


## Dataset — identical to E1 (no augmentation)

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=True, download=True, transform=transform
)
test_dataset = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=False, download=True, transform=transform
)

class_names = train_dataset.classes
train_loader, test_loader = get_dataloaders(train_dataset, test_dataset, batch_size=64)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 938, Test batches: 157


## Architecture — identical to E1 DiagnosticCNN

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def get_features(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.global_pool(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        x = self.get_features(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = DiagnosticCNN().to(device)
print(f"DiagnosticCNN params: {sum(p.numel() for p in model.parameters()):,}")

DiagnosticCNN params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Training — 40 epochs with cosine LR decay

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=40)
num_epochs = 40

train_losses = []
lr_log = []
model.train()
for epoch in range(num_epochs):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    current_lr = scheduler.get_last_lr()[0]
    lr_log.append(current_lr)
    scheduler.step()
    if epoch < 15 or (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.4f}, LR: {current_lr:.6f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')
with open(os.path.join(OUT_DIR, 'lr_schedule.txt'), 'w') as f:
    for lr in lr_log:
        f.write(f'{lr}\n')
print(f"Losses and LR schedule saved.")

Epoch [1/40], Loss: 0.4613, LR: 0.001000
Epoch [2/40], Loss: 0.2940, LR: 0.000998
Epoch [3/40], Loss: 0.2496, LR: 0.000994
Epoch [4/40], Loss: 0.2240, LR: 0.000986
Epoch [5/40], Loss: 0.2051, LR: 0.000976
Epoch [6/40], Loss: 0.1905, LR: 0.000962
Epoch [7/40], Loss: 0.1751, LR: 0.000946
Epoch [8/40], Loss: 0.1595, LR: 0.000926
Epoch [9/40], Loss: 0.1497, LR: 0.000905
Epoch [10/40], Loss: 0.1359, LR: 0.000880
Epoch [11/40], Loss: 0.1268, LR: 0.000854
Epoch [12/40], Loss: 0.1142, LR: 0.000825
Epoch [13/40], Loss: 0.1030, LR: 0.000794
Epoch [14/40], Loss: 0.0927, LR: 0.000761
Epoch [15/40], Loss: 0.0826, LR: 0.000727
Epoch [20/40], Loss: 0.0427, LR: 0.000539
Epoch [25/40], Loss: 0.0209, LR: 0.000345
Epoch [30/40], Loss: 0.0098, LR: 0.000175
Epoch [35/40], Loss: 0.0052, LR: 0.000054
Epoch [40/40], Loss: 0.0042, LR: 0.000002
Losses and LR schedule saved.


## Evaluation — identical pipeline

In [5]:
accuracy, cm, per_class = evaluate_detailed(
    model, test_loader, device, class_names, model_name='LongCNN'
)
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='LongCNN')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='LongCNN')

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write('-' * 55 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')

cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names:
        f.write(f'{name:>15}')
    f.write('\n')
    for i in range(len(class_names)):
        f.write(f'{class_names[i]:>15}')
        for j in range(len(class_names)):
            f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        true_name = class_names[c]
        total_errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {true_name}  (errors: {total_errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0:
                continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f"\nAll results saved to {OUT_DIR}/")

  Test Accuracy: 92.99%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8640     0.0148     0.8666
  Trouser             0.9810     0.0008     0.9929
  Pullover            0.9020     0.0108     0.9029
  Dress               0.9350     0.0081     0.9276
  Coat                0.8960     0.0106     0.9041
  Sandal              0.9820     0.0012     0.9889
  Shirt               0.7970     0.0233     0.7915
  Sneaker             0.9830     0.0046     0.9600
  Bag                 0.9930     0.0018     0.9841
  Ankle boot          0.9660     0.0020     0.9817

All results saved to ../outputs/error_analysis/extended_training/


## Delta vs E1 Baseline

In [6]:
def load_e1_metrics(path):
    data = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5 and parts[0] != 'Class' and '-' not in line[:5]:
                cls, roc, pr, tpr, prec = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                data[cls] = {'tpr': tpr, 'precision': prec, 'pr_auc': pr}
    return data

e1 = load_e1_metrics('../outputs/error_analysis/metrics_summary.txt')

print(f'{"Class":<15} {"E1 TPR":>8} {"E8 TPR":>8} {"Δ TPR":>8} {"E1 Prec":>8} {"E8 Prec":>8} {"Δ Prec":>8}')
print('-' * 63)
for name in class_names:
    if name in e1:
        print(f'{name:<15} {e1[name]["tpr"]:>8.3f} {per_class[name]["TPR"]:>8.3f} {per_class[name]["TPR"] - e1[name]["tpr"]:>+8.3f} {e1[name]["precision"]:>8.3f} {per_class[name]["Precision"]:>8.3f} {per_class[name]["Precision"] - e1[name]["precision"]:>+8.3f}')

print(f'\nAccuracy:  E1=92.50%  E8={accuracy:.2f}%  Δ={accuracy - 92.50:+.2f}%')
print(f'Macro PR:   E1=0.9712  E8={pr_scores["macro"]:.4f}  Δ={pr_scores["macro"] - 0.9712:+.4f}')

e8_shirt_err = cm_np[6].sum() - cm_np[6, 6]
e8_tshirt_err = cm_np[0].sum() - cm_np[0, 0]
e8_upper = e8_shirt_err + e8_tshirt_err + (cm_np[2].sum()-cm_np[2,2]) + (cm_np[4].sum()-cm_np[4,4]) + (cm_np[3].sum()-cm_np[3,3])
print(f'\nUpper-body total errors: E1=649, E8={e8_upper}')
print(f'Shirt errors:   E1=153, E8={e8_shirt_err}')
print(f'T-shirt errors: E1=163, E8={e8_tshirt_err}')

# Convergence check: compare last 5 vs first 5 loss
final_avg = np.mean(train_losses[-5:])
initial_avg = np.mean(train_losses[:5])
print(f'\nConvergence: initial avg={initial_avg:.4f} → final avg={final_avg:.4f}')
print(f'Still decreasing: {"YES" if train_losses[-1] < train_losses[-5] else "FLAT/UP"}')

Class             E1 TPR   E8 TPR    Δ TPR  E1 Prec  E8 Prec   Δ Prec
---------------------------------------------------------------
Trouser            0.988    0.981   -0.007    0.990    0.993   +0.003
Pullover           0.890    0.902   +0.012    0.896    0.903   +0.007
Dress              0.907    0.935   +0.028    0.940    0.928   -0.012
Coat               0.870    0.896   +0.026    0.931    0.904   -0.027
Sandal             0.978    0.982   +0.004    0.990    0.989   -0.001
Shirt              0.847    0.797   -0.050    0.723    0.791   +0.069
Sneaker            0.991    0.983   -0.008    0.946    0.960   +0.014
Bag                0.986    0.993   +0.007    0.982    0.984   +0.002

Accuracy:  E1=92.50%  E8=92.99%  Δ=+0.49%
Macro PR:   E1=0.9712  E8=0.9731  Δ=+0.0019

Upper-body total errors: E1=649, E8=606
Shirt errors:   E1=153, E8=203
T-shirt errors: E1=163, E8=136

Convergence: initial avg=0.2868 → final avg=0.0042
Still decreasing: YES
